# Company Brochure

### Business Challenge

- Create a product that provides the content for a company's brochure for prospective clients, investor and potential recruits.
- You will be given the company's title and primary website address

In [ ]:
# imports
import os
import requests
import json
from IPython.display import display, Markdown
from typing import Optional
from dotenv import load_dotenv
from bs4 import BeautifulSoup, Tag
from openai import OpenAI

In [ ]:
#constants
MODEL = "gpt-4o-mini"

In [ ]:
# website_url: Optional[str] = input("Enter the company's website URL: ")
website_url: Optional[str] = "https://zephbyte.com/"

if not website_url or not website_url.startswith("http"):
    raise ValueError("Invalid URL. Please enter a valid website URL starting with 'http'.")

In [ ]:
# load environment variables
load_dotenv()
openai_api_key = os.getenv("OPEN_AI_KEY")
if not openai_api_key:
    raise ValueError("OPEN_AI_KEY not found in environment variables.")
elif openai_api_key and not openai_api_key.startswith("sk-proj-"):
    raise ValueError("Invalid OPEN_AI_KEY.")

In [ ]:
# web scrapper

class WebScraper:
    """A simple web scraper that fetches the content of a webpage and extracts its title."""
    url: str
    title: Optional[str] = None
    body: bytes = b""
    content: str = ""
    links: list[str] = []

    def __init__(self, url: str):
        self.url = url
        self._fetch_content()

    def _fetch_content(self):
        response = requests.get(self.url)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        # Safely assign title as a string or None
        self.title = soup.title.string if soup.title and soup.title.string else "No Title Found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input", "button"]):
                irrelevant.decompose()
            self.content = soup.body.get_text(separator="\n", strip=True)

        links: list[str] = [
            href for link in soup.find_all("a", href=True)
            if isinstance(link, Tag) and isinstance((href := link.get("href")) ,str) and href.startswith("http")
        ]
        self.links = links

    def get_content(self) -> str:
        """Returns the content of the webpage."""
        return f"Website Title: {self.title}\n\n Website Content: \n\n{self.content}\n\n"

### Let GPT-4o-mini figure out the relevant links

In [ ]:
system_prompt: str = "You are helful assistant that helps user to create a company brochure. \
    You will be given the content of the company's website and links found on the company's website. \
    You will use the relevant links to gather the information about the links and use the content of the website to create a company brochure. \
    You can use links like About Page, Contact Page, Services, Careers, etc but you will not use links like Privacy Policy, Terms of Service, etc."

system_prompt += "\n\nYou should response in JSON format as the following structure:\n"
system_prompt += """{
    links: [
        {type: "about page", "full_url": "https://www.example.com/about"},
        {type: "careers page", "full_url": "https://www.example.com/careers"}
    ]
}"""

**User Prompt**

In [ ]:
def get_user_prompt(website: WebScraper) -> str:
    """Generates the user prompt for the OpenAI API."""
    user_prompt =  f"""
    Here is the list of links found on the company's website: {website.url} -
    Please decide which links are relevant to create a company brochure and which links are not relevant.
    Response with full URL of the links and do not include links that are not relevant, e.g., Privacy Policy, Terms of Service, etc.
    \nLinks (some might be relevant):
    """

    user_prompt += "\n\nYou should response in JSON format as the following structure:\n"
    user_prompt += """{
        links: [
            {type: "about page", "full_url": "https://www.example.com/about"},
            {type: "careers page", "full_url": "https://www.example.com/careers"}
        ]
    }"""
    user_prompt += '\n'.join(website.links)
    return user_prompt

### Step 1: User OpenAI to get relevent Links

In [ ]:
def get_relevant_links(website_url: str) -> dict[str, list[dict[str, str]]]:
    """Uses OpenAI API to get the relevant links from the website."""
    website: WebScraper = WebScraper(website_url)
    messages = [{
        "role" : "system",
        "content": system_prompt
    }, {
        "role" : "user",
        "content": get_user_prompt(website)
    }]
    try:
        openai_client = OpenAI(api_key=openai_api_key)
        completion = openai_client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0.7,
            response_format={"type": "json_object"}
        )
        response: Optional[str] = completion.choices[0].message.content
        if not response:
            raise ValueError("No response from OpenAI API.")
    except Exception as e:
        raise RuntimeError(f"Error while calling OpenAI API: {e}")
    return json.loads(response)

### Step 2: Make the company's brochure

In [ ]:
# get details of each of the relevant links
def get_complete_detail(url: str) -> str:
    result = "Landing Page: \n"
    result += WebScraper(url).get_content()
    relevant_links = get_relevant_links(url)
    if not relevant_links or "links" not in relevant_links:
        raise ValueError("No relevant links found.")
    
    for link in relevant_links["links"]:
        result += f"\n\n{link['type']}\n"
        result += WebScraper(link["full_url"]).get_content()
    return result

In [ ]:
system_prompt = "You are a helpful assistant that helps user to create a company brochure. \
You will be given the content from the relevant links from the company's website. \
You will use the content to create a company brochure about the company perspective, services, prospective custoemrs and recruits. \
Response in Markdown. Include company's culture, values, careers/jobs(if you have the information) and mission statement."

In [ ]:
def create_company_brochure_prompt(company_name: str, url: str) -> str:
    """Generates the prompt for creating a company brochure."""
    user_prompt =  f"You are looking at the content of the company called {company_name}."
    user_prompt += f"\n\n Here is the content for the landing page and other relevent pages of the website {url}.\n"
    user_prompt += "\n\nUse this information to create the short company brochure in Markdown format."
    user_prompt += get_complete_detail(url)
    user_prompt = user_prompt[:20_000]
    return user_prompt

In [ ]:
def strip_markdown_code_block(md: str) -> str:
    """Remove leading and trailing triple backticks and 'markdown' from a Markdown code block."""
    lines = md.strip().splitlines()
    if lines and lines[0].startswith("```"):
        # Remove the first line (```markdown or ```)
        lines = lines[1:]
    if lines and lines[-1].startswith("```"):
        # Remove the last line (```)
        lines = lines[:-1]
    return "\n".join(lines)

def create_brochure(company_name:str, url: str):
    """Creates a company brochure using OpenAI API."""
    try:
        openai_client = OpenAI(api_key=openai_api_key)
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": create_company_brochure_prompt(company_name, url),
            },
        ]
        openai_client = OpenAI(api_key=openai_api_key)
        completion = openai_client.chat.completions.create(
            model=MODEL,
            messages=messages    
        )
        response: Optional[str] = completion.choices[0].message.content
        if not response:
            raise ValueError("No response from OpenAI API.")
    except Exception as e:
        raise RuntimeError(f"Error while calling OpenAI API: {e}")
    
    response = strip_markdown_code_block(response)
    display(Markdown(response))

In [ ]:
create_brochure("Zephbyte", website_url)